# Znajdywanie kolorów chwytaka

## Tworzenie dataframe

In [1]:
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
import cv2

In [2]:
SAVE_DIR = Path("outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
masks_array = np.load(str(SAVE_DIR)+'/masks_array.npy')
incomplete_detect = np.load(str(SAVE_DIR)+'/incomplete_detect.npy')
df = pd.read_csv(str(SAVE_DIR)+'/df.csv')
incomplete_detect

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  21,  28,  34,
        42,  49,  57,  58,  59,  66,  68,  70,  72,  88,  89,  90,  91,
        96, 103, 104, 108, 111, 112, 113, 114, 115, 116, 117, 118, 119,
       120, 123, 152])

### (x,y) z GDSAM

In [4]:
xy_array = []

for i in range(0, masks_array.shape[0]):

  # Załadowanie obrazu głębokości
  depth_img = Image.open(df['depth_path'][i]).convert("RGB")
  depth_img = np.asarray(depth_img)
  depth_img = depth_img[:, :, 0]

  tmp = []

  for j in range(0, masks_array.shape[1]):

    # Utworzenie maski
    mask = (masks_array[i][j] * depth_img)
  
    # Obliczanie momentów
    moments = cv2.moments(mask)

    # Wysokość maski
    y_indices, x_indices = np.where(mask > 0)
    if len(y_indices) == 0 or len(x_indices) == 0:
      tmp.extend([0, 0,])
      continue
    mask_height = np.max(y_indices) - np.min(y_indices) + 1
    mask_width = np.max(x_indices) - np.min(x_indices) + 1

    # Momenty centralne
    m10 = moments['m10']
    m01 = moments['m01']
    m00 = moments['m00']

    # Środki elementów
    if m00 == 0:
      tmp.extend([0, 0,])
    else:
      x = m10/m00
      y = m01/m00
      #z = np.mean(mask[mask > 0])
      tmp.extend([x, y])
  xy_array.append(tmp)

xy_array = np.array(xy_array)
print(f"xy_array shape -> {xy_array.shape}")

xy_array shape -> (172, 6)


In [5]:
xy_array = np.delete(xy_array, incomplete_detect, axis=0)
xy_array.shape

(130, 6)

### (x,y) z LLM

#### DLA OPENAI

In [6]:
import base64
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

Kodowanie base64 do API openai

In [7]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")
    
encoded_images = df["color_path"].apply(encode_image).to_numpy().reshape(-1,1)

encoded_images = np.delete(encoded_images, incomplete_detect, axis=0)
encoded_images.shape

(130, 1)

In [8]:
xy_array_gpt = []
for i in range(0, 10):#encoded_images.shape[0]):
    base64_image = encoded_images[i][0]
    
    prompt_text = (
        "The image shows a robot gripper with three colored elements: pink, green, and blue. "
        "Your task is to detect each element and provide normalized bounding box coordinates "
        "in the exact format: (x_min,y_min,x_max,y_max),(x_min,y_min,x_max,y_max),(x_min,y_min,x_max,y_max) "
        "for pink, green, blue respectively. "
        "Coordinates must be floating-point numbers in [0,1] range, relative to image width and height "
        "(origin at top-left corner). "
        "Return ONLY the coordinates in this format, no additional text, explanations, or comments."
    )

    response = client.responses.create(
        model="gpt-5",
        input=[
            {
                "role": "user",
                "content": [
                    { "type": "input_text", "text": prompt_text },
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "high"
                    },
                ],
            }
        ],
    )
    xy_array_gpt.append(response.output_text)

print(xy_array_gpt[:2])

['(0.414,0.321,0.465,0.443),(0.480,0.278,0.520,0.358),(0.541,0.316,0.592,0.443)', '(0.580,0.320,0.610,0.460),(0.600,0.250,0.640,0.340),(0.640,0.310,0.680,0.460)']


In [9]:
len(xy_array_gpt)

10

#### DLA GOOGLE

In [10]:
import dotenv
dotenv.load_dotenv()

True

In [11]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

models = [m.name for m in genai.list_models()
          if "generateContent" in getattr(m, "supported_generation_methods", [])]
print([m for m in models if "robot" in m.lower() or "er" in m.lower() or "1.5" in m.lower()])

model = genai.GenerativeModel('models/gemini-robotics-er-1.5-preview')

/home/ruszczka/projekty/test_files/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


['models/gemini-robotics-er-1.5-preview', 'models/gemini-2.5-computer-use-preview-10-2025']


/home/ruszczka/projekty/test_files/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
encoded_images = df["color_path"].to_numpy().reshape(-1,1)

encoded_images = np.delete(encoded_images, incomplete_detect, axis=0)
encoded_images.shape

(130, 1)

#### KONWERSJA

In [13]:
xy_array_gemini = []
for i in range(0, 10):
    image_path = encoded_images[i][0]
    img = Image.open(image_path)
    w, h = img.size

    prompt_text = (
        "The image shows a robot gripper with three colored elements: pink, green, and blue. "
        "Your task is to detect each element and provide normalized bounding box coordinates "
        "in the exact format: (x_min,y_min,x_max,y_max),(x_min,y_min,x_max,y_max),(x_min,y_min,x_max,y_max) "
        "for pink, green, blue respectively. "
        "Coordinates must be floating-point numbers in [0,1] range, relative to image width and height "
        "(origin at top-left corner). "
        "Return ONLY the coordinates in this format, no additional text, explanations, or comments."
    )

    resp = model.generate_content([prompt_text, img])
    xy_array_gemini.append(resp.text.strip())

print(xy_array_gemini[:2])

['(0.39,0.23,0.43,0.28),(0.46,0.24,0.50,0.27),(0.53,0.28,0.57,0.32)', '(0.54,0.28,0.60,0.35),(0.58,0.27,0.63,0.33),(0.62,0.28,0.67,0.37)']


In [14]:
xy_array_gemini = np.array(xy_array_gemini).reshape(-1,1)
xy_array_gemini.shape

(10, 1)

In [17]:
#np.save(str(SAVE_DIR)+'/xy_array_gemini2.npy', xy_array_gemini)
#np.save(str(SAVE_DIR)+'/xy_array_gpt2.npy', xy_array_gpt)
xy_array_gpt = np.load(str(SAVE_DIR)+'/xy_array_gpt2.npy')
xy_array_gemini = np.load(str(SAVE_DIR)+'/xy_array_gemini2.npy')

#### IMPORT GOTOWYCH

In [ ]:
#xy_array_gpt = np.load(str(SAVE_DIR)+'/xy_array_gpt.npy')
#xy_array_gemini = np.load(str(SAVE_DIR)+'/xy_array_gemini.npy')

In [ ]:
import re

def parse_xy_rows(arr):
    out = []
    for row in arr:
        text = row[0] if isinstance(row, (list, np.ndarray)) else str(row)
        pairs = re.findall(r'\(([0-9]*\.?[0-9]+)\s*,\s*([0-9]*\.?[0-9]+)\)', text)
        nums = [float(x) for p in pairs for x in p]
        # docięcie/padding do 6 liczb
        if len(nums) < 6:
            nums += [0.0] * (6 - len(nums))
        else:
            nums = nums[:6]
        out.append(nums)
    return np.array(out, dtype=float)

In [1]:
xy_array_openai_num = parse_xy_rows(xy_array_gpt)
xy_array_gemini_num = parse_xy_rows(xy_array_gemini)

print("OpenAI num shape:", xy_array_openai_num.shape)
print("Gemini num shape:", xy_array_gemini_num.shape)

NameError: name 'parse_xy_rows' is not defined

## Wizualizacja

In [20]:
from pathlib import Path
import matplotlib.pyplot as plt
import cv2

In [21]:
id_array = np.delete(np.arange(172), incomplete_detect, axis=0)
id_array.shape

(130,)

In [ ]:
# Utworzenie folderu na ploty
plots_dir = Path("plots_comparison/centers")
plots_dir.mkdir(exist_ok=True)

num_to_plot = min(10, len(xy_array), len(xy_array_openai_num), len(xy_array_gemini_num), len(id_array))

for nr_xy in range(num_to_plot):
    fig, axes = plt.subplots(1, 3, figsize=(36, 12))

    # Wczytaj obraz raz
    img_bgr = cv2.imread(df['color_path'][id_array[nr_xy]])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # 1) GDSAM (piksele)
    axes[0].imshow(img_rgb)
    axes[0].scatter(xy_array[nr_xy][0], xy_array[nr_xy][1], color='red', marker='x', s=60)
    axes[0].scatter(xy_array[nr_xy][2], xy_array[nr_xy][3], color='red', marker='x', s=60)
    axes[0].scatter(xy_array[nr_xy][4], xy_array[nr_xy][5], color='red', marker='x', s=60)
    axes[0].set_title(f'GDSAM (#{nr_xy})', fontsize=16)
    axes[0].axis('off')

    # 2) OpenAI (norm -> px)
    axes[1].imshow(img_rgb)
    axes[1].scatter(xy_array_openai_num[nr_xy][0]*w, xy_array_openai_num[nr_xy][1]*h, color='red', marker='x', s=60)
    axes[1].scatter(xy_array_openai_num[nr_xy][2]*w, xy_array_openai_num[nr_xy][3]*h, color='red', marker='x', s=60)
    axes[1].scatter(xy_array_openai_num[nr_xy][4]*w, xy_array_openai_num[nr_xy][5]*h, color='red', marker='x', s=60)
    axes[1].set_title('OpenAI', fontsize=16)
    axes[1].axis('off')

    # 3) Google (norm -> px)
    axes[2].imshow(img_rgb)
    axes[2].scatter(xy_array_gemini_num[nr_xy][0]*w, xy_array_gemini_num[nr_xy][1]*h, color='red', marker='x', s=60)
    axes[2].scatter(xy_array_gemini_num[nr_xy][2]*w, xy_array_gemini_num[nr_xy][3]*h, color='red', marker='x', s=60)
    axes[2].scatter(xy_array_gemini_num[nr_xy][4]*w, xy_array_gemini_num[nr_xy][5]*h, color='red', marker='x', s=60)
    axes[2].set_title('Google Gemini', fontsize=16)
    axes[2].axis('off')

    plt.tight_layout()
    out = plots_dir / f"comparison_plot_triplet_{nr_xy:02d}.png"
    plt.savefig(out, dpi=300, bbox_inches='tight')
    print(f"Zapisano: {out}")
    plt.show()

print(f"\nWszystkie ploty zostały zapisane w folderze: {plots_dir}")

# Fine-tune konfiguracja robota

In [ ]:
import pandas as pd
from pathlib import Path
import base64
import numpy as np

In [ ]:
SAVE_DIR = Path("outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

incomplete_detect = np.load(str(SAVE_DIR)+'/incomplete_detect.npy')
df = pd.read_csv(str(SAVE_DIR)+'/df.csv')
incomplete_detect

In [ ]:
df = df.drop(incomplete_detect)

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [ ]:
df["color_path"] = df["color_path"].apply(encode_image)
df = df.drop(columns=["depth_path", "time", "id"])

In [ ]:
df

In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
from sklearn.model_selection import train_test_split
import json

output_columns = [
    "ESJoint1", "ESJoint2", "ESJoint3", "ESJoint4", "ESJoint5", "ESJoint6",
    "gripper_finger_1_joint", "gripper_finger_2_joint"
]

# Podział na train/val/test
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=0)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=0)

def save_jsonl_text(dataframe, filename):
    with open(filename, "w") as f:
        for idx, row in dataframe.iterrows():
            user_content = f"Obraz chwytaka robota (base64): {row['color_path']}\nPodaj konfigurację robota."
            config = ", ".join(f"{col}: {row[col]}" for col in output_columns)
            assistant_content = f"Konfiguracja robota: {config}"
            example = {
                "messages": [
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": assistant_content}
                ]
            }
            f.write(json.dumps(example, ensure_ascii=False) + "\n")

save_jsonl_text(train_df, "train_text.jsonl")
save_jsonl_text(val_df, "val_text.jsonl")
save_jsonl_text(test_df, "test_text.jsonl")

In [ ]:
from sklearn.model_selection import train_test_split
import json
import base64

output_columns = [
    "ESJoint1", "ESJoint2", "ESJoint3", "ESJoint4", "ESJoint5", "ESJoint6",
    "gripper_finger_1_joint", "gripper_finger_2_joint"
]

# Przygotowanie danych do Vision fine-tuning
# Wczytaj dane ponownie bez kodowania base64
df_vision = pd.read_csv(str(SAVE_DIR)+'/df.csv')
df_vision = df_vision.drop(incomplete_detect)
df_vision = df_vision.drop(columns=["depth_path", "time", "id"])

# Podział na train/val/test dla Vision
train_df_vision, temp_df_vision = train_test_split(df_vision, test_size=0.3, random_state=0)
val_df_vision, test_df_vision = train_test_split(temp_df_vision, test_size=0.5, random_state=0)

def encode_image_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def save_jsonl_vision(dataframe, filename):
    with open(filename, "w") as f:
        for idx, row in dataframe.iterrows():
            # Koduj obraz do base64
            base64_image = encode_image_base64(row['color_path'])
            config = ", ".join(f"{col}: {row[col]}" for col in output_columns)
            example = {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": "Podaj konfigurację robota na podstawie obrazu chwytaka."},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                        ]
                    },
                    {
                        "role": "assistant",
                        "content": f"Konfiguracja robota: {config}"
                    }
                ]
            }
            f.write(json.dumps(example, ensure_ascii=False) + "\n")
            if idx % 10 == 0:
                print(config)

# Generuj pliki JSONL dla Vision fine-tuning
save_jsonl_vision(train_df_vision, "train_vision.jsonl")
save_jsonl_vision(val_df_vision, "val_vision.jsonl")
save_jsonl_vision(test_df_vision, "test_vision.jsonl")

print("Pliki JSONL dla Vision fine-tuning zostały utworzone.")
print(f"Train: {len(train_df_vision)} przykładów")
print(f"Val: {len(val_df_vision)} przykładów") 
print(f"Test: {len(test_df_vision)} przykładów")

In [ ]:
# Przygotowanie plików dla OpenAI Vision fine-tuning
import shutil
import zipfile

# Utwórz katalog dla obrazów  
images_dir = Path("vision_finetune_images")
images_dir.mkdir(exist_ok=True)

# Skopiuj wszystkie obrazy do jednego katalogu
all_image_paths = set()
for df_split in [train_df_vision, val_df_vision, test_df_vision]:
    for _, row in df_split.iterrows():
        src_path = row['color_path']
        dst_path = images_dir / Path(src_path).name
        if not dst_path.exists():  # Skopiuj tylko jeśli nie istnieje
            shutil.copy2(src_path, dst_path)
        all_image_paths.add(src_path)

print(f"Skopiowano {len(all_image_paths)} unikalnych obrazów do {images_dir}")

# Utwórz kompletne archiwum ZIP z obrazami i plikami JSONL
with zipfile.ZipFile("vision_finetune_complete.zip", 'w') as zipf:
    # Dodaj pliki JSONL
    zipf.write("train_vision.jsonl")
    zipf.write("val_vision.jsonl") 
    zipf.write("test_vision.jsonl")
    
    # Dodaj wszystkie obrazy do katalogu "images/" w ZIP
    for image_file in images_dir.glob("*"):
        if image_file.is_file():
            zipf.write(image_file, f"images/{image_file.name}")

print("Utworzono vision_finetune_complete.zip")

In [ ]:
# Upload plików do OpenAI i uruchomienie fine-tuningu przez Python API
import os
from openai import OpenAI

# Inicjalizacja klienta OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

try:
    # Upload training file
    print("Przesyłanie pliku treningowego...")
    with open("train_vision.jsonl", "rb") as f:
        train_file = client.files.create(file=f, purpose="fine-tune")
    print(f"Plik treningowy przesłany: {train_file.id}")

    # Upload validation file
    print("Przesyłanie pliku walidacyjnego...")
    with open("val_vision.jsonl", "rb") as f:
        val_file = client.files.create(file=f, purpose="fine-tune")
    print(f"Plik walidacyjny przesłany: {val_file.id}")

    # Uruchom fine-tuning
    print("Uruchamianie fine-tuning...")
    fine_tune_job = client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model="gpt-4o-2024-08-06",
        suffix="robot-config"
    )

    print(f"Fine-tuning job uruchomiony!")
    print(f"Job ID: {fine_tune_job.id}")
    print(f"Status: {fine_tune_job.status}")
    print(f"Model: {fine_tune_job.model}")
    
except Exception as e:
    print(f"Błąd: {e}")
    print("Sprawdź czy:")
    print("1. Zmienna OPENAI_API_KEY jest ustawiona")
    print("2. Pliki train_vision.jsonl i val_vision.jsonl istnieją")
    print("3. Masz wystarczający balans na koncie OpenAI")

In [ ]:
# Wczytaj oryginalne dane
df = pd.read_csv("outputs/df.csv")
incomplete_detect = np.load("outputs/incomplete_detect.npy")
df_vision = df.drop(incomplete_detect).drop(columns=["depth_path", "time", "id"])

# Odtwórz podział z tym samym random_state
train_df_vision, temp_df_vision = train_test_split(df_vision, test_size=0.3, random_state=0)
val_df_vision, test_df_vision = train_test_split(temp_df_vision, test_size=0.5, random_state=0)

# Pokaż nazwy plików testowych
test_image_names = [Path(path).name for path in test_df_vision['color_path']]
print("Obrazy testowe:")
for name in test_image_names:
    print(name)